# ICD ReAct Single-Case Debug Notebook

Use this notebook to isolate where a single ICD ReAct case slows down or fails.

Suggested order:
1. Run the setup cells first.
2. Fetch one case and inspect the summary length.
3. Run the agent-only cell. If that hangs or errors, the issue is in the ReAct path rather than MLflow batch evaluation.
4. Run the optional MLflow cell only after the agent-only path is understood.

In [ ]:
import os

In [2]:
from __future__ import annotations



import json

import os

from pathlib import Path

import sys





def find_project_root(start: Path | None = None) -> Path:

    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:

        if (candidate / 'pyproject.toml').exists():

            return candidate

    raise RuntimeError('Could not locate project root from current notebook working directory.')





PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(0, str(PROJECT_ROOT))



print(json.dumps({

    'project_root': str(PROJECT_ROOT),

    'cwd': str(Path.cwd().resolve()),

    'python': sys.executable,

    'exec_env': os.environ.get('EXECUTION_ENV'),

}, indent=2))


{
  "project_root": "C:\\Users\\sonutka\\main\\1-Projects\\manual_deterministic_executor",
  "cwd": "C:\\Users\\sonutka\\main\\1-Projects\\manual_deterministic_executor\\baselines\\icd_react",
  "python": "c:\\Users\\sonutka\\main\\2-Envs\\tam-env\\Scripts\\python.exe",
  "exec_env": "local"
}


In [3]:
import json

import time

import traceback



import mlflow

import mlflow.langchain

import pandas as pd



from baselines.icd_react import load_config

from baselines.icd_react.runtime import (

    resolve_mlflow_experiment,

    resolve_model_name,

    resolve_spark_session,

    resolve_strict_table,

)

from baselines.icd_react.single_case import (

    coerce_icd_code_list,

    fetch_case_record,

    run_single_case_prediction,

    score_prediction,

)

from baselines.icd_react.tools import open_tabular_entry

from scripts.evaluate_icd_react import load_case_selectors



print('Imports OK')


c:\Users\sonutka\main\2-Envs\tam-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [4]:
import json



USE_FIRST_SELECTOR = True

TARGET_HADM_ID = None

TARGET_SUBJECT_ID = None

TARGET_NOTE_ID = None

MANUAL_SUMMARY = None

MANUAL_EXPECTED_CODES = []

MODEL_NAME_OVERRIDE = None

RUN_MLFLOW_TRACE = True

MAX_AGENT_STEPS = 40

STREAM_PREVIEW_CHARS = 1200



config = load_config()

strict_table = resolve_strict_table()

model_name = MODEL_NAME_OVERRIDE or resolve_model_name()

mlflow_experiment = resolve_mlflow_experiment()



print(json.dumps({

    'execution_env': config.execution_env,

    'manuals_root': str(config.manuals_root),

    'strict_table': strict_table,

    'model_name': model_name,

    'mlflow_experiment': mlflow_experiment,

    'use_first_selector': USE_FIRST_SELECTOR,

    'max_agent_steps': MAX_AGENT_STEPS,

    'stream_preview_chars': STREAM_PREVIEW_CHARS,

}, indent=2))


{
  "execution_env": "local",
  "manuals_root": "C:\\Users\\sonutka\\main\\1-Projects\\manual_deterministic_executor\\data\\ICD-2019-manual",
  "strict_table": "usdo_aa_catalog.research_tam_datasets.mimic_icd10_note_dataset_2017_2019_strict",
  "model_name": "openai:gpt-5",
  "mlflow_experiment": "/Users/sonutka@mfcgd.com/icd-react-single-case",
  "use_first_selector": true,
  "max_agent_steps": 40,
  "stream_preview_chars": 1200
}


In [20]:
import time



spark = None

if MANUAL_SUMMARY is None:

    start = time.perf_counter()

    spark = resolve_spark_session(app_name='icd-react-single-case-debug-notebook')

    print(f'Spark ready in {time.perf_counter() - start:.2f}s')

else:

    print('Skipping Spark session because MANUAL_SUMMARY is set.')


Spark ready in 1.28s


In [6]:
if MANUAL_SUMMARY:
    case_record = {
        'hadm_id': None,
        'subject_id': None,
        'note_id': None,
        'case_summary': str(MANUAL_SUMMARY).strip(),
        'expected_icd_codes': coerce_icd_code_list(MANUAL_EXPECTED_CODES),
    }
    selector = None
else:
    if USE_FIRST_SELECTOR and TARGET_HADM_ID is None and TARGET_SUBJECT_ID is None and TARGET_NOTE_ID is None:
        selector = load_case_selectors(spark=spark, table_name=strict_table, limit=1)[0]
    else:
        selector = {
            'hadm_id': TARGET_HADM_ID,
            'subject_id': TARGET_SUBJECT_ID,
            'note_id': TARGET_NOTE_ID,
        }

    case_record = fetch_case_record(
        spark=spark,
        table_name=strict_table,
        hadm_id=int(selector['hadm_id']) if selector.get('hadm_id') is not None else None,
        subject_id=int(selector['subject_id']) if selector.get('subject_id') is not None else None,
        note_id=str(selector['note_id']) if selector.get('note_id') is not None else None,
    )

case_metadata = {
    'hadm_id': case_record.get('hadm_id'),
    'subject_id': case_record.get('subject_id'),
    'note_id': case_record.get('note_id'),
    'expected_icd_code_count': len(case_record.get('expected_icd_codes', [])),
    'expected_icd_codes': case_record.get('expected_icd_codes', []),
    'summary_length': len(case_record.get('case_summary') or ''),
}

pd.DataFrame([case_metadata])

,hadm_id,subject_id,note_id,expected_icd_code_count,expected_icd_codes,summary_length
0,20000147,14990224,14990224-DS-7,18,"[I213, I5021, E1121, E1165, I25119, I110, E113...",10303


In [7]:
summary_text = case_record.get('case_summary') or ''
print(f'Summary length: {len(summary_text)} characters')
print()
print(summary_text[:4000])

Summary length: 10303 characters

Name:  ___                    Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   M
 
Service: CARDIOTHORACIC
 
Allergies: 
Influenza Virus Vaccines / simvastatin / Crestor
 
Attending: ___.
 
Chief Complaint:
chest pain 
 
Major Surgical or Invasive Procedure:
___ Cardiac catheterization 
___ Emergent coronary artery bypass grafting x4 with
the left internal mammary artery to the left anterior
descending artery and reverse saphenous vein graft to the
ramus intermedius artery, and a sequential vein graft to the
___ and ___ obtuse marginal artery

 
History of Present Illness:
___ year old male with a history of
diabetes and hyperlipidemia who is non complaints with
medications who had chest pain last week that radiated to his
back. 
 He presented today to ___ with complaints of chest pain
that awoke him from sleep at 3am. He described it as heartburn
and radiated to his back and arm. On pres

In [8]:
valid_lookup = open_tabular_entry(config, 'I21.3')
invalid_lookup = open_tabular_entry(config, 'F32.A')

print('Valid lookup sample:')
print(json.dumps({
    'code': valid_lookup.get('code'),
    'description': valid_lookup.get('description'),
    'section': valid_lookup.get('section'),
}, indent=2))
print()
print('Invalid lookup sample:')
print(json.dumps(invalid_lookup, indent=2))

Valid lookup sample:
{
  "code": "I21.3",
  "description": "ST elevation (STEMI) myocardial infarction of unspecified site",
  "section": "Ischemic heart diseases (I20-I25)"
}

Invalid lookup sample:
{
  "manual": "ICD-10-CM Tabular",
  "requested_code": "F32.A",
  "normalized_code": "F32A",
  "found": false,
  "message": "Tabular code not found. Inspect a parent code or one of the nearby suggestions instead of assuming this code exists in the 2019 ICD manual.",
  "suggestions": []
}


In [17]:
from baselines.icd_react.agent import build_icd_agent

from baselines.icd_react.single_case import (

    build_single_case_prompt,

    extract_json_object,

    message_content_to_text,

    normalize_prediction,

    serialize_agent_messages,

)



agent_result = None

agent_error = None

agent_elapsed_seconds = None

stream_steps = []

stream_last_state = None



agent = build_icd_agent(model=model_name, config=config)

prompt_efficiency_suffix = (
    "\n\nEfficiency guidance:\n"
    "- If you have several candidate ICD codes to verify, issue the needed tool calls in the same turn.\n"
    "- If you have already inspected a tabular entry for a candidate code, do not reopen that same code unless conflicting evidence appears.\n"
    "- Do not reopen the same guideline unless new evidence makes it necessary.\n"
    "- Keep the final rationale concise and keep supporting_evidence to the shortest useful list."
)

agent_inputs = {

    'messages': [

        {

            'role': 'user',

            'content': build_single_case_prompt(summary_text) + prompt_efficiency_suffix,

        }

    ]

}



start = time.perf_counter()

try:

    for step_number, state in enumerate(

        agent.stream(

            agent_inputs,

            config={'recursion_limit': MAX_AGENT_STEPS},

            stream_mode='values',

        ),

        start=1,

    ):

        stream_last_state = state

        messages = state.get('messages', [])

        latest_message = messages[-1] if messages else None

        message_type = latest_message.__class__.__name__ if latest_message is not None else 'None'

        text = '' if latest_message is None else message_content_to_text(getattr(latest_message, 'content', ''))

        tool_calls = [] if latest_message is None else list(getattr(latest_message, 'tool_calls', None) or [])

        message_name = None if latest_message is None else getattr(latest_message, 'name', None)



        step_record = {

            'step': step_number,

            'message_type': message_type,

            'message_name': message_name,

            'tool_call_count': len(tool_calls),

            'text_preview': text[:STREAM_PREVIEW_CHARS],

        }

        stream_steps.append(step_record)



        print(f'\n=== Agent Step {step_number} | {message_type} ===')

        if message_name:

            print(f'name: {message_name}')

        if tool_calls:

            print('tool_calls:')

            print(json.dumps(tool_calls, indent=2, default=str))

        if text:

            preview = text[:STREAM_PREVIEW_CHARS]

            print(preview)

            if len(text) > STREAM_PREVIEW_CHARS:

                print('\n[truncated]')

        if not text and not tool_calls:

            print('[no visible text content]')



    if stream_last_state is None:

        raise RuntimeError('Agent stream returned no state updates.')



    final_messages = stream_last_state.get('messages', [])

    if not final_messages:

        raise RuntimeError('Agent stream finished without any messages.')



    final_message = final_messages[-1]

    final_message_text = message_content_to_text(getattr(final_message, 'content', ''))

    prediction = normalize_prediction(extract_json_object(final_message_text))

    agent_result = {

        'prediction': prediction,

        'messages': serialize_agent_messages(final_messages),

        'final_message_text': final_message_text,

        'stream_steps': stream_steps,

    }

except Exception as exc:

    agent_error = exc

    traceback.print_exc()

finally:

    agent_elapsed_seconds = time.perf_counter() - start



print(f'\nAgent elapsed: {agent_elapsed_seconds:.2f}s')

if agent_error is None:

    print(json.dumps(agent_result['prediction'], indent=2))

    print()

    print('Message count:', len(agent_result.get('messages', [])))

    print('Visible step count:', len(agent_result.get('stream_steps', [])))

else:

    print(f'Agent error: {type(agent_error).__name__}: {agent_error}')


=== Agent Step 1 | HumanMessage ===
You are reviewing one clinical case for ICD-10-CM diagnosis coding.

Task:
- Use the ICD manual tools to navigate the Index, Tabular List, and Guidelines as needed.
- Do not finalize a code until you inspect the Tabular entry.
- Return the final ICD-10-CM diagnosis codes you would actually assign, not just high-level category headings.
- Do not invent extra child-code specificity that is not supported by the case summary and the manual text you inspected.
- Be conservative and avoid unsupported codes.
- Return only valid JSON matching this schema exactly.

Schema:
{
  "predicted_icd_codes": ["CODE1", "CODE2"],
  "confidence": 0.0,
  "rationale": "short explanation",
  "supporting_evidence": ["short evidence item 1", "short evidence item 2"]
}

Clinical case summary:
Name:  ___                    Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   M
 
Service: CARDIOTHORACIC
 
Allergies: 


In [18]:
if agent_result is None:
    print('No metrics because the agent cell did not produce a result.')
else:
    expected_codes = case_record.get('expected_icd_codes', [])
    predicted_codes = agent_result['prediction'].get('predicted_icd_codes', [])
    expected_set = set(expected_codes)
    predicted_set = set(predicted_codes)

    comparison = {
        'expected_icd_codes': expected_codes,
        'predicted_icd_codes': predicted_codes,
        'correct_predictions': sorted(expected_set & predicted_set),
        'missed_expected_codes': sorted(expected_set - predicted_set),
        'unexpected_predicted_codes': sorted(predicted_set - expected_set),
    }

    comparison_df = pd.DataFrame(
        [
            {
                'code': code,
                'expected': code in expected_set,
                'predicted': code in predicted_set,
            }
            for code in sorted(expected_set | predicted_set)
        ]
    )

    print('Expected vs predicted:')
    print(json.dumps(comparison, indent=2))
    print()
    display(comparison_df)
    print()

    metrics = score_prediction(
        predicted_codes=predicted_codes,
        expected_codes=expected_codes,
    )
    print('Metrics:')
    print(json.dumps(metrics, indent=2))

Expected vs predicted:
{
  "expected_icd_codes": [
    "I213",
    "I5021",
    "E1121",
    "E1165",
    "I25119",
    "I110",
    "E113299",
    "E7800",
    "M549",
    "F329",
    "E785",
    "I340",
    "Z794",
    "Z9114",
    "N400",
    "M179",
    "K429",
    "G5601"
  ],
  "predicted_icd_codes": [
    "I2102",
    "I5021",
    "I2510",
    "Z951",
    "E1165",
    "Z794",
    "I10",
    "E7800",
    "N400",
    "F329",
    "M179",
    "K429",
    "R809",
    "G5601",
    "G8929"
  ],
  "correct_predictions": [
    "E1165",
    "E7800",
    "F329",
    "G5601",
    "I5021",
    "K429",
    "M179",
    "N400",
    "Z794"
  ],
  "missed_expected_codes": [
    "E1121",
    "E113299",
    "E785",
    "I110",
    "I213",
    "I25119",
    "I340",
    "M549",
    "Z9114"
  ],
  "unexpected_predicted_codes": [
    "G8929",
    "I10",
    "I2102",
    "I2510",
    "R809",
    "Z951"
  ]
}



,code,expected,predicted
0,E1121,True,False
1,E113299,True,False
2,E1165,True,True
3,E7800,True,True
4,E785,True,False
5,F329,True,True
6,G5601,True,True
7,G8929,False,True
8,I10,False,True
9,I110,True,False



Metrics:
{
  "predicted_count": 15,
  "expected_count": 18,
  "true_positive_count": 9,
  "precision": 0.6,
  "recall": 0.5
}


In [21]:
from datetime import UTC, datetime
from importlib import reload
import json
import time
import traceback

import baselines.icd_react.single_case as single_case_module

single_case_module = reload(single_case_module)
run_single_case_prediction = single_case_module.run_single_case_prediction
score_prediction = single_case_module.score_prediction
fetch_case_record = single_case_module.fetch_case_record

PROMPT_SWEEP_VARIANTS = {
    'baseline': '',
    'same_turn': (
        'Efficiency guidance:\n'
        '- If you have several candidate ICD codes to verify, issue the needed tool calls in the same turn.\n'
        '- Keep the final rationale concise and keep supporting_evidence to the shortest useful list.'
    ),
    'no_repeat_tabular_guideline': (
        'Efficiency guidance:\n'
        '- If you have several candidate ICD codes to verify, issue the needed tool calls in the same turn.\n'
        '- If you have already inspected a tabular entry for a candidate code, do not reopen that same code unless conflicting evidence appears.\n'
        '- Do not reopen the same guideline unless new evidence makes it necessary.\n'
        '- Keep the final rationale concise and keep supporting_evidence to the shortest useful list.'
    ),
    'active_first_no_repeat': (
        'Efficiency guidance:\n'
        '- If you have several candidate ICD codes to verify, issue the needed tool calls in the same turn.\n'
        '- Prioritize diagnoses that were evaluated, treated, monitored, or clearly affected this admission before revisiting lower-yield background history.\n'
        '- If you have already inspected a tabular entry for a candidate code, do not reopen that same code unless conflicting evidence appears.\n'
        '- Do not reopen the same guideline unless new evidence makes it necessary.\n'
        '- Keep the final rationale concise and keep supporting_evidence to the shortest useful list.'
    ),
}
PROMPT_SWEEP_CASE_LIMIT = 10
PROMPT_SWEEP_STAGE1_CASES = 3
PROMPT_SWEEP_FINALISTS = 2
prompt_sweep_output_path = PROJECT_ROOT / 'docs' / 'icd_prompt_search_notebook_live.json'
prompt_sweep_rows_by_variant = {name: [] for name in PROMPT_SWEEP_VARIANTS}
prompt_sweep_ranked_stage1 = []
prompt_sweep_finalists = []
prompt_sweep_started_at = datetime.now(UTC).isoformat()
prompt_sweep_best_variant = None


def _score_summary(rows):
    ok_rows = [row for row in rows if row.get('status') == 'ok']
    if not ok_rows:
        return {
            'case_count': len(rows),
            'ok_case_count': 0,
            'error_case_count': len(rows),
            'mean_precision': 0.0,
            'mean_recall': 0.0,
            'mean_f1': 0.0,
            'mean_runtime_seconds': 0.0,
        }
    precisions = [float(row.get('precision', 0.0) or 0.0) for row in ok_rows]
    recalls = [float(row.get('recall', 0.0) or 0.0) for row in ok_rows]
    f1s = [float(row.get('f1', 0.0) or 0.0) for row in ok_rows]
    runtimes = [float(row.get('elapsed_seconds', 0.0) or 0.0) for row in ok_rows]
    return {
        'case_count': len(rows),
        'ok_case_count': len(ok_rows),
        'error_case_count': len(rows) - len(ok_rows),
        'mean_precision': round(sum(precisions) / len(precisions), 6),
        'mean_recall': round(sum(recalls) / len(recalls), 6),
        'mean_f1': round(sum(f1s) / len(f1s), 6),
        'mean_runtime_seconds': round(sum(runtimes) / len(runtimes), 3),
    }


def _rank_variants():
    ranked = []
    for variant_name, rows in prompt_sweep_rows_by_variant.items():
        ranked.append({'variant': variant_name, **_score_summary(rows)})
    ranked.sort(
        key=lambda item: (
            float(item.get('mean_f1', 0.0)),
            float(item.get('mean_recall', 0.0)),
            -float(item.get('mean_runtime_seconds', 0.0)),
        ),
        reverse=True,
    )
    return ranked


def _write_checkpoint(status):
    payload = {
        'status': status,
        'started_at': prompt_sweep_started_at,
        'finished_at': datetime.now(UTC).isoformat(),
        'case_limit': PROMPT_SWEEP_CASE_LIMIT,
        'stage1_cases': PROMPT_SWEEP_STAGE1_CASES,
        'finalists': prompt_sweep_finalists,
        'model_name': model_name,
        'strict_table': strict_table,
        'variant_suffixes': PROMPT_SWEEP_VARIANTS,
        'stage1_ranked': prompt_sweep_ranked_stage1,
        'final_ranked': _rank_variants(),
        'rows_by_variant': prompt_sweep_rows_by_variant,
    }
    prompt_sweep_output_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    return payload


def _evaluate_variant(variant_name, selectors):
    prompt_suffix = PROMPT_SWEEP_VARIANTS[variant_name]
    for case_index, selector in enumerate(selectors, start=1):
        row = {
            'variant': variant_name,
            'hadm_id': selector.get('hadm_id'),
            'subject_id': selector.get('subject_id'),
            'note_id': selector.get('note_id'),
            'status': 'error',
            'error': '',
        }
        print(json.dumps({'event': 'case_start', 'variant': variant_name, 'case_index': case_index, 'selector': row}, default=str))
        started = time.perf_counter()
        try:
            case_record = fetch_case_record(
                spark=spark,
                table_name=strict_table,
                hadm_id=int(selector['hadm_id']) if selector.get('hadm_id') is not None else None,
                subject_id=int(selector['subject_id']) if selector.get('subject_id') is not None else None,
                note_id=str(selector['note_id']) if selector.get('note_id') is not None else None,
            )
            result = run_single_case_prediction(
                config=config,
                summary_text=str(case_record.get('case_summary') or ''),
                model_name=model_name,
                prompt_suffix=prompt_suffix,
            )
            metrics = score_prediction(
                predicted_codes=result['prediction'].get('predicted_icd_codes', []),
                expected_codes=case_record.get('expected_icd_codes', []),
            )
            precision = float(metrics.get('precision', 0.0) or 0.0)
            recall = float(metrics.get('recall', 0.0) or 0.0)
            f1 = (2.0 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
            row.update({
                'status': 'ok',
                'expected_icd_codes': case_record.get('expected_icd_codes', []),
                'predicted_icd_codes': result['prediction'].get('predicted_icd_codes', []),
                **metrics,
                'f1': round(f1, 6),
            })
        except Exception as exc:
            row['error'] = f'{type(exc).__name__}: {exc}'
            traceback.print_exc()
        finally:
            row['elapsed_seconds'] = round(time.perf_counter() - started, 3)
        prompt_sweep_rows_by_variant[variant_name].append(row)
        _write_checkpoint('running')
        print(json.dumps({'event': 'case_end', 'variant': variant_name, 'status': row['status'], 'elapsed_seconds': row['elapsed_seconds'], 'precision': row.get('precision'), 'recall': row.get('recall'), 'error': row.get('error')}, default=str))


prompt_sweep_selectors = load_case_selectors(
    spark=spark,
    table_name=strict_table,
    limit=PROMPT_SWEEP_CASE_LIMIT,
)
if not prompt_sweep_selectors:
    raise RuntimeError('No ICD cases matched the prompt sweep selection.')

prompt_sweep_stage1_selectors = prompt_sweep_selectors[:PROMPT_SWEEP_STAGE1_CASES]
prompt_sweep_stage2_selectors = prompt_sweep_selectors[PROMPT_SWEEP_STAGE1_CASES:PROMPT_SWEEP_CASE_LIMIT]

print(json.dumps({
    'event': 'search_started',
    'started_at': prompt_sweep_started_at,
    'case_limit': PROMPT_SWEEP_CASE_LIMIT,
    'stage1_cases': len(prompt_sweep_stage1_selectors),
    'stage2_cases': len(prompt_sweep_stage2_selectors),
    'variants': list(PROMPT_SWEEP_VARIANTS.keys()),
    'model_name': model_name,
    'strict_table': strict_table,
    'output_path': str(prompt_sweep_output_path),
}, indent=2))

for variant_name in PROMPT_SWEEP_VARIANTS:
    _evaluate_variant(variant_name, prompt_sweep_stage1_selectors)

prompt_sweep_ranked_stage1 = _rank_variants()
prompt_sweep_finalists = [item['variant'] for item in prompt_sweep_ranked_stage1[:PROMPT_SWEEP_FINALISTS]]
_write_checkpoint('stage1_complete')
print(json.dumps({'event': 'stage1_complete', 'ranked_stage1': prompt_sweep_ranked_stage1, 'finalists': prompt_sweep_finalists}, indent=2))

for variant_name in prompt_sweep_finalists:
    _evaluate_variant(variant_name, prompt_sweep_stage2_selectors)

prompt_sweep_final_ranked = _rank_variants()
prompt_sweep_best_variant = prompt_sweep_final_ranked[0]['variant'] if prompt_sweep_final_ranked else None
_write_checkpoint('completed')
print(json.dumps({'event': 'search_complete', 'best_variant': prompt_sweep_best_variant, 'final_ranked': prompt_sweep_final_ranked, 'output_path': str(prompt_sweep_output_path)}, indent=2))

{
  "event": "search_started",
  "started_at": "2026-05-15T01:21:17.737704+00:00",
  "case_limit": 10,
  "stage1_cases": 3,
  "stage2_cases": 7,
  "variants": [
    "baseline",
    "same_turn",
    "no_repeat_tabular_guideline",
    "active_first_no_repeat"
  ],
  "model_name": "openai:gpt-5",
  "strict_table": "usdo_aa_catalog.research_tam_datasets.mimic_icd10_note_dataset_2017_2019_strict",
  "output_path": "C:\\Users\\sonutka\\main\\1-Projects\\manual_deterministic_executor\\docs\\icd_prompt_search_notebook_live.json"
}
{"event": "case_start", "variant": "baseline", "case_index": 1, "selector": {"variant": "baseline", "hadm_id": 20000147, "subject_id": 14990224, "note_id": "14990224-DS-7", "status": "error", "error": ""}}
{"event": "case_end", "variant": "baseline", "status": "ok", "elapsed_seconds": 373.119, "precision": 0.571429, "recall": 0.222222, "error": ""}
{"event": "case_start", "variant": "baseline", "case_index": 2, "selector": {"variant": "baseline", "hadm_id": 20000343,

In [22]:
PROMPT_SWEEP_BACKFILL_VARIANTS = ['baseline']

for variant_name in PROMPT_SWEEP_BACKFILL_VARIANTS:
    completed_count = len(prompt_sweep_rows_by_variant.get(variant_name, []))
    remaining_selectors = prompt_sweep_selectors[completed_count:PROMPT_SWEEP_CASE_LIMIT]
    print(json.dumps({
        'event': 'backfill_start',
        'variant': variant_name,
        'completed_count': completed_count,
        'remaining_cases': len(remaining_selectors),
    }, indent=2))
    if remaining_selectors:
        _evaluate_variant(variant_name, remaining_selectors)

prompt_sweep_final_ranked = _rank_variants()
prompt_sweep_best_variant = prompt_sweep_final_ranked[0]['variant'] if prompt_sweep_final_ranked else None
_write_checkpoint('completed')
print(json.dumps({
    'event': 'backfill_complete',
    'best_variant': prompt_sweep_best_variant,
    'final_ranked': prompt_sweep_final_ranked,
    'output_path': str(prompt_sweep_output_path),
}, indent=2))

{
  "event": "backfill_start",
  "variant": "baseline",
  "completed_count": 3,
  "remaining_cases": 7
}
{"event": "case_start", "variant": "baseline", "case_index": 1, "selector": {"variant": "baseline", "hadm_id": 20000769, "subject_id": 11082580, "note_id": "11082580-DS-14", "status": "error", "error": ""}}
{"event": "case_end", "variant": "baseline", "status": "ok", "elapsed_seconds": 378.782, "precision": 0.25, "recall": 0.066667, "error": ""}
{"event": "case_start", "variant": "baseline", "case_index": 2, "selector": {"variant": "baseline", "hadm_id": 20001363, "subject_id": 13346796, "note_id": "13346796-DS-8", "status": "error", "error": ""}}
{"event": "case_end", "variant": "baseline", "status": "ok", "elapsed_seconds": 192.045, "precision": 1.0, "recall": 0.111111, "error": ""}
{"event": "case_start", "variant": "baseline", "case_index": 3, "selector": {"variant": "baseline", "hadm_id": 20001770, "subject_id": 10117812, "note_id": "10117812-DS-13", "status": "error", "error":

In [27]:
if not RUN_MLFLOW_TRACE:
    print('Set RUN_MLFLOW_TRACE = True in the config cell before running this cell.')
else:
    start = time.perf_counter()
    if mlflow.active_run() is not None:
        mlflow.end_run()
    mlflow.set_experiment(mlflow_experiment)
    mlflow.langchain.autolog(log_traces=True, silent=True)
    print(f'MLflow configured in {time.perf_counter() - start:.2f}s')

    with mlflow.start_run(run_name='icd_react_single_case_debug_notebook') as run:
        with mlflow.start_span(name='single_case_icd_react_debug') as case_span:
            case_span.set_inputs({
                'hadm_id': case_record.get('hadm_id'),
                'subject_id': case_record.get('subject_id'),
                'note_id': case_record.get('note_id'),
                'model_name': model_name,
                'summary_length': len(summary_text),
            })
            traced_start = time.perf_counter()
            traced_result = run_single_case_prediction(
                config=config,
                summary_text=summary_text,
                model_name=model_name,
            )
            traced_elapsed = time.perf_counter() - traced_start
            case_span.set_outputs({
                'status': 'ok',
                'predicted_icd_codes': traced_result['prediction'].get('predicted_icd_codes', []),
                'elapsed_seconds': traced_elapsed,
            })
        mlflow.log_dict(traced_result['prediction'], 'single_case_debug/prediction.json')
        print(json.dumps({
            'run_id': run.info.run_id,
            'elapsed_seconds': traced_elapsed,
            'prediction': traced_result['prediction'],
        }, indent=2))

MLflow configured in 3.36s
{
  "run_id": "1c4c101160b64b119420d9deda08d39a",
  "elapsed_seconds": 373.7510262999858,
  "prediction": {
    "predicted_icd_codes": [
      "I213",
      "I5021",
      "I2510",
      "Z951",
      "E1165",
      "Z794",
      "I10",
      "E7800"
    ],
    "confidence": 0.73,
    "rationale": "Patient had an acute ST-elevation MI treated during this admission with emergent CABG for multivessel coronary artery disease and developed acute systolic heart failure. Diabetes type 2 was uncontrolled with hyperglycemia and insulin was prescribed at discharge. Hypertension and hypercholesterolemia were present and managed. MI site was not explicitly documented by the provider, so a conservative unspecified-site STEMI code was selected. Other listed chronic conditions lacked evidence of evaluation or treatment during the stay, so were not coded.",
    "supporting_evidence": [
      "I21.3 Tabular: ST elevation (STEMI) myocardial infarction of unspecified site; dis

In [ ]:
## Trace Analysis



Use these cells after a successful traced run to inspect where the time went.



The goal is to confirm whether a slow case is dominated by tool runtime, model runtime, or growing model context across turns.


In [28]:
import json

TRACE_ID = 'tr-1d5419db477f88add9e34b52214e41c4'
TRACE_LINK = 'https://adb-2498687920111167.7.azuredatabricks.net/ml/experiments/2273108800187843/evaluation-runs?o=2498687920111167&selectedRunUuid=1c4c101160b64b119420d9deda08d39a&selectedEvaluationId=tr-1d5419db477f88add9e34b52214e41c4'

print(json.dumps({
    'trace_id': TRACE_ID,
    'trace_link': TRACE_LINK,
}, indent=2))


{
  "trace_id": "tr-1d5419db477f88add9e34b52214e41c4",
  "trace_link": "https://adb-2498687920111167.7.azuredatabricks.net/ml/experiments/2273108800187843/evaluation-runs?o=2498687920111167&selectedRunUuid=1c4c101160b64b119420d9deda08d39a&selectedEvaluationId=tr-1d5419db477f88add9e34b52214e41c4"
}


In [29]:
import json

import mlflow
import pandas as pd


def _decode_maybe_json(value):
    if value is None:
        return None
    if isinstance(value, (dict, list, int, float, bool)):
        return value
    if not isinstance(value, str):
        return value
    try:
        return json.loads(value)
    except Exception:
        return value


def _json_size(value):
    if value is None:
        return 0
    try:
        return len(json.dumps(value, default=str))
    except Exception:
        return len(str(value))


def _pick(mapping, *keys):
    if not isinstance(mapping, dict):
        return None
    for key in keys:
        value = mapping.get(key)
        if value is not None:
            return value
    return None


trace_obj = mlflow.get_trace(TRACE_ID)
trace_dict = trace_obj.to_dict()
trace_info = trace_dict.get('info', {})
raw_spans = trace_dict.get('data', {}).get('spans', [])
parent_by_id = {span.get('span_id'): span for span in raw_spans}

rows = []
for order, span in enumerate(sorted(raw_spans, key=lambda item: item.get('start_time_unix_nano', 0)), start=1):
    attrs = span.get('attributes', {})
    span_type = _decode_maybe_json(attrs.get('mlflow.spanType')) or 'UNKNOWN'
    inputs = _decode_maybe_json(attrs.get('mlflow.spanInputs'))
    outputs = _decode_maybe_json(attrs.get('mlflow.spanOutputs'))
    metadata = _decode_maybe_json(attrs.get('metadata')) or {}
    token_usage = _decode_maybe_json(attrs.get('mlflow.chat.tokenUsage')) or {}
    parent_span = parent_by_id.get(span.get('parent_span_id'))
    parent_name = None if parent_span is None else parent_span.get('name')
    duration_seconds = (span.get('end_time_unix_nano', 0) - span.get('start_time_unix_nano', 0)) / 1_000_000_000

    rows.append(
        {
            'order': order,
            'span_id': span.get('span_id'),
            'parent_span_id': span.get('parent_span_id'),
            'parent_name': parent_name,
            'name': span.get('name'),
            'span_type': span_type,
            'duration_seconds': duration_seconds,
            'start_time_unix_nano': span.get('start_time_unix_nano'),
            'end_time_unix_nano': span.get('end_time_unix_nano'),
            'status_code': ((span.get('status') or {}).get('code')),
            'input_chars': _json_size(inputs),
            'output_chars': _json_size(outputs),
            'prompt_tokens': _pick(token_usage, 'input_tokens', 'prompt_tokens'),
            'completion_tokens': _pick(token_usage, 'output_tokens', 'completion_tokens'),
            'total_tokens': _pick(token_usage, 'total_tokens'),
            'langgraph_step': _pick(metadata, 'langgraph_step'),
            'langgraph_node': _pick(metadata, 'langgraph_node'),
            'inputs': inputs,
            'outputs': outputs,
            'metadata': metadata,
        }
    )

spans_df = pd.DataFrame(rows)
trace_summary = {
    'trace_id': trace_info.get('trace_id', TRACE_ID),
    'span_count': len(spans_df),
    'execution_duration_seconds': (trace_info.get('execution_duration_ms') or 0) / 1000,
    'request_preview_chars': len(str(trace_info.get('request_preview') or '')),
    'response_preview_chars': len(str(trace_info.get('response_preview') or '')),
}

print(json.dumps(trace_summary, indent=2))
spans_df[['order', 'name', 'span_type', 'duration_seconds', 'input_chars', 'output_chars']].head(10)


{
  "trace_id": "tr-1d5419db477f88add9e34b52214e41c4",
  "span_count": 80,
  "execution_duration_seconds": 373.754,
  "request_preview_chars": 128,
  "response_preview_chars": 146
}


,order,name,span_type,duration_seconds,input_chars,output_chars
0,1,single_case_icd_react_debug,UNKNOWN,373.754605,128,146
1,2,icd_manual_agent,CHAIN,373.708606,11462,118662
2,3,model,CHAIN,24.769054,11573,1392
3,4,ChatOpenAI,CHAT_MODEL,24.748045,12106,308
4,5,tools,CHAIN,0.004501,138,9469
5,6,list_index_main_terms_tool,TOOL,0.001001,28,9419
6,7,model,CHAIN,10.135021,22349,1370
7,8,ChatOpenAI,CHAT_MODEL,10.121012,21684,289
8,9,tools,CHAIN,0.005502,123,3669
9,10,list_tabular_chapters_tool,TOOL,0.002197,13,3619


In [30]:
type_summary = (
    spans_df.groupby('span_type', dropna=False)
    .agg(
        span_count=('span_id', 'count'),
        total_seconds=('duration_seconds', 'sum'),
        mean_seconds=('duration_seconds', 'mean'),
        max_seconds=('duration_seconds', 'max'),
        total_input_chars=('input_chars', 'sum'),
        total_output_chars=('output_chars', 'sum'),
    )
    .sort_values('total_seconds', ascending=False)
)

type_summary['share_of_trace'] = type_summary['total_seconds'] / max(trace_summary['execution_duration_seconds'], 1e-9)

type_summary


,span_count,total_seconds,mean_seconds,max_seconds,total_input_chars,total_output_chars,share_of_trace
span_type,,,,,,,
CHAIN,40,751.605431,18.790136,373.708606,936062,227095,2.010963
UNKNOWN,1,373.754605,373.754605,373.754605,128,146,1.000002
CHAT_MODEL,14,371.259439,26.518531,62.052660,805356,7613,0.993326
TOOL,25,3.887149,0.155486,0.768566,560,84614,0.010400


In [31]:
slow_spans_df = spans_df.sort_values('duration_seconds', ascending=False).copy()
slow_spans_df[
    [
        'order',
        'name',
        'span_type',
        'parent_name',
        'duration_seconds',
        'input_chars',
        'output_chars',
        'prompt_tokens',
        'completion_tokens',
        'total_tokens',
        'langgraph_step',
        'langgraph_node',
    ]
].head(20)


,order,name,span_type,parent_name,duration_seconds,input_chars,output_chars,prompt_tokens,completion_tokens,total_tokens,langgraph_step,langgraph_node
0,1,single_case_icd_react_debug,UNKNOWN,None,373.754605,128,146,NaN,NaN,NaN,NaN,None
1,2,icd_manual_agent,CHAIN,single_case_icd_react_debug,373.708606,11462,118662,NaN,NaN,NaN,NaN,None
78,79,model,CHAIN,icd_manual_agent,62.070188,115733,3000,NaN,NaN,NaN,27.0,model
79,80,ChatOpenAI,CHAT_MODEL,model,62.052660,99373,1889,28040.0,4668.0,32708.0,27.0,model
10,11,model,CHAIN,icd_manual_agent,42.795488,27303,2870,NaN,NaN,NaN,5.0,model
11,12,ChatOpenAI,CHAT_MODEL,model,42.779481,25443,2070,7983.0,3227.0,11210.0,5.0,model
70,71,model,CHAIN,icd_manual_agent,40.759474,111205,1385,NaN,NaN,NaN,23.0,model
71,72,ChatOpenAI,CHAT_MODEL,model,40.745480,97253,292,27625.0,2974.0,30599.0,23.0,model
38,39,model,CHAIN,icd_manual_agent,39.094093,47986,1384,NaN,NaN,NaN,7.0,model
39,40,ChatOpenAI,CHAT_MODEL,model,39.079517,43619,293,11968.0,3358.0,15326.0,7.0,model


In [32]:
model_spans_df = spans_df[spans_df['span_type'] == 'CHAT_MODEL'].sort_values('order').copy()
tool_spans_df = spans_df[spans_df['span_type'] == 'TOOL'].sort_values('order').copy()

llm_total_seconds = float(model_spans_df['duration_seconds'].sum()) if not model_spans_df.empty else 0.0
tool_total_seconds = float(tool_spans_df['duration_seconds'].sum()) if not tool_spans_df.empty else 0.0

if model_spans_df.empty:
    print('No CHAT_MODEL spans found in this trace.')
else:
    sample_n = min(5, len(model_spans_df))
    early_model_spans = model_spans_df.head(sample_n)
    late_model_spans = model_spans_df.tail(sample_n)
    growth_summary = pd.DataFrame(
        [
            {
                'window': 'early_model_calls',
                'count': len(early_model_spans),
                'avg_duration_seconds': early_model_spans['duration_seconds'].mean(),
                'avg_input_chars': early_model_spans['input_chars'].mean(),
                'avg_output_chars': early_model_spans['output_chars'].mean(),
                'avg_total_tokens': early_model_spans['total_tokens'].dropna().mean() if not early_model_spans['total_tokens'].dropna().empty else None,
            },
            {
                'window': 'late_model_calls',
                'count': len(late_model_spans),
                'avg_duration_seconds': late_model_spans['duration_seconds'].mean(),
                'avg_input_chars': late_model_spans['input_chars'].mean(),
                'avg_output_chars': late_model_spans['output_chars'].mean(),
                'avg_total_tokens': late_model_spans['total_tokens'].dropna().mean() if not late_model_spans['total_tokens'].dropna().empty else None,
            },
        ]
    )

    diagnosis = {
        'trace_total_seconds': trace_summary['execution_duration_seconds'],
        'llm_total_seconds': llm_total_seconds,
        'tool_total_seconds': tool_total_seconds,
        'llm_share_of_trace': llm_total_seconds / max(trace_summary['execution_duration_seconds'], 1e-9),
        'tool_share_of_trace': tool_total_seconds / max(trace_summary['execution_duration_seconds'], 1e-9),
        'model_call_count': len(model_spans_df),
        'tool_call_count': len(tool_spans_df),
        'later_calls_have_larger_inputs': bool(late_model_spans['input_chars'].mean() > early_model_spans['input_chars'].mean()),
        'later_calls_have_longer_duration': bool(late_model_spans['duration_seconds'].mean() > early_model_spans['duration_seconds'].mean()),
    }

    print(json.dumps(diagnosis, indent=2))
    print()
    display(growth_summary)
    print()
    display(
        model_spans_df[
            [
                'order',
                'name',
                'duration_seconds',
                'input_chars',
                'output_chars',
                'prompt_tokens',
                'completion_tokens',
                'total_tokens',
                'parent_name',
            ]
        ]
    )


{
  "trace_total_seconds": 373.754,
  "llm_total_seconds": 371.25943929999994,
  "tool_total_seconds": 3.8871488,
  "llm_share_of_trace": 0.9933256615313814,
  "tool_share_of_trace": 0.010400286819672831,
  "model_call_count": 14,
  "tool_call_count": 25,
  "later_calls_have_larger_inputs": true,
  "later_calls_have_longer_duration": true
}



,window,count,avg_duration_seconds,avg_input_chars,avg_output_chars,avg_total_tokens
0,early_model_calls,5,27.009090,29591.4,657.6,10752.0
1,late_model_calls,5,29.331066,89865.6,612.6,27454.4


,order,name,duration_seconds,input_chars,output_chars,prompt_tokens,completion_tokens,total_tokens,parent_name
3,4,ChatOpenAI,24.748045,12106,308,4017.0,1823.0,5840.0,model
7,8,ChatOpenAI,10.121012,21684,289,7018.0,604.0,7622.0,model
11,12,ChatOpenAI,42.779481,25443,2070,7983.0,3227.0,11210.0,model
39,40,ChatOpenAI,39.079517,43619,293,11968.0,3358.0,15326.0,model
43,44,ChatOpenAI,18.317394,45105,328,12319.0,1443.0,13762.0,model
47,48,ChatOpenAI,34.074000,45624,317,12404.0,2784.0,15188.0,model
51,52,ChatOpenAI,1.950073,46310,317,12543.0,30.0,12573.0,model
55,56,ChatOpenAI,33.216935,57804,317,15824.0,2597.0,18421.0,model
59,60,ChatOpenAI,20.317649,58333,311,15914.0,1631.0,17545.0,model
63,64,ChatOpenAI,2.384129,59286,298,16142.0,26.0,16168.0,model


In [33]:
FOCUS_SPAN_ORDER = int(model_spans_df.sort_values('duration_seconds', ascending=False).iloc[0]['order']) if not model_spans_df.empty else None

if FOCUS_SPAN_ORDER is None:
    print('No model span available to inspect.')
else:
    focus_row = model_spans_df.loc[model_spans_df['order'] == FOCUS_SPAN_ORDER].iloc[0]
    print(json.dumps({
        'focus_span_order': int(focus_row['order']),
        'name': focus_row['name'],
        'duration_seconds': float(focus_row['duration_seconds']),
        'input_chars': int(focus_row['input_chars']),
        'output_chars': int(focus_row['output_chars']),
        'prompt_tokens': None if pd.isna(focus_row['prompt_tokens']) else int(focus_row['prompt_tokens']),
        'completion_tokens': None if pd.isna(focus_row['completion_tokens']) else int(focus_row['completion_tokens']),
        'total_tokens': None if pd.isna(focus_row['total_tokens']) else int(focus_row['total_tokens']),
        'parent_name': focus_row['parent_name'],
    }, indent=2))
    print('\nInput preview:\n')
    print(json.dumps(focus_row['inputs'], indent=2, default=str)[:6000])
    print('\n\nOutput preview:\n')
    print(json.dumps(focus_row['outputs'], indent=2, default=str)[:4000])


{
  "focus_span_order": 80,
  "name": "ChatOpenAI",
  "duration_seconds": 62.0526601,
  "input_chars": 99373,
  "output_chars": 1889,
  "prompt_tokens": 28040,
  "completion_tokens": 4668,
  "total_tokens": 32708,
  "parent_name": "model"
}

Input preview:

{
  "messages": [
    {
      "role": "system",
      "content": "You are an ICD-10-CM manual navigation agent.\n\nWork like a careful manual user, not like a shortcut classifier.\n\nRules:\n- Use the provided manual tools to inspect the ICD Index, Tabular List, and Guidelines.\n- Do not assume a diagnosis code is valid until you inspect the Tabular entry.\n- Use the Guidelines only when a general coding rule appears relevant.\n- Prefer explicit manual evidence over intuition.\n- If the manual path is uncertain, say so rather than inventing certainty.",
      "tool_calls": null,
      "refusal": null,
      "tool_call_id": null
    },
    {
      "role": "user",
      "content": "You are reviewing one clinical case for ICD-10-CM dia

### Tool Calls vs Completion Tokens

This section shows why a model turn can still have many completion tokens even when it is "just making tool calls".

OpenAI counts the assistant's emitted tool-call payloads, including function names and JSON arguments, as output tokens. Final natural-language answers are also completion tokens.


In [ ]:
def _extract_choice_message(outputs):
    if not isinstance(outputs, dict):
        return {}
    choices = outputs.get('choices') or []
    if not choices:
        return {}
    first_choice = choices[0] or {}
    return first_choice.get('message') or {}


def _tool_args_chars(tool_calls):
    total = 0
    for tool_call in tool_calls or []:
        function_payload = tool_call.get('function') or {}
        total += len(str(function_payload.get('arguments') or ''))
        total += len(str(function_payload.get('name') or ''))
    return total


model_output_rows = []
for _, row in model_spans_df.iterrows():
    message = _extract_choice_message(row['outputs'])
    tool_calls = message.get('tool_calls') or []
    content = message.get('content') or ''
    finish_reason = None
    outputs = row['outputs'] if isinstance(row['outputs'], dict) else {}
    choices = outputs.get('choices') or []
    if choices:
        finish_reason = (choices[0] or {}).get('finish_reason')

    model_output_rows.append(
        {
            'order': int(row['order']),
            'duration_seconds': float(row['duration_seconds']),
            'prompt_tokens': None if pd.isna(row['prompt_tokens']) else int(row['prompt_tokens']),
            'completion_tokens': None if pd.isna(row['completion_tokens']) else int(row['completion_tokens']),
            'total_tokens': None if pd.isna(row['total_tokens']) else int(row['total_tokens']),
            'tool_call_count': len(tool_calls),
            'tool_argument_chars': _tool_args_chars(tool_calls),
            'content_chars': len(content),
            'has_text_content': bool(content),
            'has_tool_calls': bool(tool_calls),
            'finish_reason': finish_reason,
            'tool_names': [((tool_call.get('function') or {}).get('name')) for tool_call in tool_calls],
            'content_preview': content[:1000],
        }
    )

model_output_df = pd.DataFrame(model_output_rows)
print('CHAT_MODEL outputs in trace order:')
display(
    model_output_df[
        [
            'order',
            'duration_seconds',
            'prompt_tokens',
            'completion_tokens',
            'tool_call_count',
            'tool_argument_chars',
            'content_chars',
            'has_text_content',
            'has_tool_calls',
        ]
    ]
)

print('\nSorted by completion_tokens descending:')
display(
    model_output_df.sort_values('completion_tokens', ascending=False)[
        [
            'order',
            'duration_seconds',
            'completion_tokens',
            'tool_call_count',
            'tool_argument_chars',
            'content_chars',
            'tool_names',
        ]
    ]
)


In [ ]:
EXAMPLE_ORDERS = list(model_output_df.sort_values('completion_tokens', ascending=False)['order'].head(3))

for order in EXAMPLE_ORDERS:
    row = model_output_df.loc[model_output_df['order'] == order].iloc[0]
    span_row = model_spans_df.loc[model_spans_df['order'] == order].iloc[0]
    message = _extract_choice_message(span_row['outputs'])
    print(f'\n=== CHAT_MODEL ORDER {order} ===')
    print(json.dumps({
        'duration_seconds': row['duration_seconds'],
        'completion_tokens': row['completion_tokens'],
        'tool_call_count': row['tool_call_count'],
        'tool_argument_chars': row['tool_argument_chars'],
        'content_chars': row['content_chars'],
        'tool_names': row['tool_names'],
    }, indent=2))
    print('\nAssistant message preview:\n')
    print(json.dumps(message, indent=2, default=str)[:5000])
